# Your First Trading Bot in ~30 Lines of Python

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mindgaptech/algodrill-notebooks/blob/main/notebooks/first_trading_bot.ipynb)

Companion notebook to [algodrill.app/code/first-trading-bot](https://algodrill.app/code/first-trading-bot). A complete, runnable SMA crossover strategy: fetch ten years of SPY daily data, compute two moving averages, backtest with realistic fees, and read the output honestly. The bot underperforms buy-and-hold -- that gap is the lesson.

**Nothing here is investment advice.** This is a backtest, not a live trading system, and it never connects to a broker. See [algodrill.app/backtesting-pitfalls](https://algodrill.app/backtesting-pitfalls) before drawing conclusions from any backtest, including this one.

## Setup

Run this cell first if you're in Colab (installs the pinned versions below into the Colab runtime). If you're running locally with the versions already installed, skip it.

In [1]:
# Colab only -- uncomment to install. Local runs should already have these
# pinned via: uv venv .venv && uv pip install -p .venv vectorbt==1.0.0 yfinance==1.4.1 pandas==2.3.3
# !pip install -q vectorbt==1.0.0 yfinance==1.4.1 pandas==2.3.3

## The strategy

10-day SMA crossing above a 50-day SMA (golden cross) enters long; the death cross exits. $10,000 starting cash, 5 bps fees per side, fixed date range (2015-01-01 to 2024-12-31) for reproducibility.

In [2]:
"""AlgoDrill -- your first trading bot in ~30 lines.

A complete, runnable strategy: fetch data -> compute signals -> backtest ->
read the stats. Backtest only -- this script never touches a broker.

Versions: vectorbt 1.0.0 . yfinance 1.4.1 . pandas 2.3.3
"""
import vectorbt as vbt

# 1. Data -- ten years of SPY daily closes (fixed range = reproducible)
price = vbt.YFData.download(
    "SPY", start="2015-01-01", end="2024-12-31"
).get("Close")

# 2. Signals -- 10-day fast SMA crossing a 50-day slow SMA
fast = vbt.MA.run(price, 10)
slow = vbt.MA.run(price, 50)
entries = fast.ma_crossed_above(slow)   # golden cross -> buy
exits = fast.ma_crossed_below(slow)     # death cross  -> sell

# 3. Backtest -- $10,000 start, 5 bps fees per side (cost realism matters),
#    freq="d" so Sharpe annualizes daily bars correctly
pf = vbt.Portfolio.from_signals(
    price, entries, exits, init_cash=10_000, fees=0.0005, freq="d"
)

# 4. The numbers that matter
print(f"Total Return    {pf.total_return():>8.1%}")
print(f"Sharpe Ratio    {pf.sharpe_ratio():>8.2f}")
print(f"Max Drawdown    {pf.max_drawdown():>8.1%}")
print(f"Trades          {pf.trades.count():>5}")
print(f"Win Rate        {pf.trades.win_rate():>8.1%}")

Total Return      106.6%
Sharpe Ratio        0.87
Max Drawdown      -15.1%
Trades             30
Win Rate           46.7%


## Plot the equity curve

Not in the original script (the page's walkthrough is text-only) -- added here because a notebook is the natural place for a chart.

In [3]:
pf.value().vbt.plot().show()

## Reading the numbers honestly

The strategy returns roughly **+107%** over 2015-2024 against a buy-and-hold SPY position that returns roughly **+241%** over the same window (see the printed output above for the exact run). The strategy underperforms by a wide margin -- that is not a bug, it is the whole point of the exercise. This simple bot reduces max drawdown from SPY's roughly -34% (2020 peak-to-trough) to about -15%, at the cost of a lot of upside. Whether that trade-off is worth it is a risk-tolerance question, not a technical one.

Two pages are essential background before reading momentum or SMA mechanics further:

- [Backtesting Pitfalls](https://algodrill.app/backtesting-pitfalls) -- look-ahead bias, survivorship bias, overfitting, and why this backtest already has one structural problem (in-sample optimization of the 10/50 windows).
- [Kelly Criterion](https://algodrill.app/kelly-criterion) -- given a sub-50% win rate, how much capital should each trade risk?

Full line-by-line walkthrough: [algodrill.app/code/first-trading-bot](https://algodrill.app/code/first-trading-bot). More runnable examples: [algodrill.app/code](https://algodrill.app/code).